In [1]:
# Scientific stack
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

# External models
from NN_UTCI import NN_UTCI


In [13]:
?NN_UTCI

Signature:
NN_UTCI(
    Ta: Union[float, numpy.ndarray, pandas.core.series.Series],
    Tr: Union[float, numpy.ndarray, pandas.core.series.Series],
    va: Union[float, numpy.ndarray, pandas.core.series.Series],
    rH: Union[float, numpy.ndarray, pandas.core.series.Series],
    oob: str = 'nan',
) -> numpy.ndarray
Docstring:
Calculate UTCI using a pre-trained neural network approximator as described in Pastine et al.
NN is trained on the same data as the polynomial approximation in Bröde et al. (2012) and covers the same input domain.

Accepts scalars, numpy arrays, or pandas Series for all meterological inputs.

Parameters
----------
Ta  : Air temperature (°C),          valid range: -50 to +50
Tr  : Mean radiant temperature (°C), valid range: Ta-80 to Ta+120
va  : Wind speed at 10 m (m/s),      valid range: 0.5 to 30.3
rH  : Relative humidity (%),         valid range: 5 to 100
oob : Out-of-bounds handling strategy:
        "nan"   – return NaN for any out-of-bounds row (default)
    

## Test on the data from Brode et al

In [2]:
esm4 = pd.read_csv('C:\\Users\\nerc-user\\OneDrive - Nexus365\\UTCI_NN\\Main\\data\\ESM_4_Table_Offset.DAT', sep='\t', comment='*')
test = pd.read_csv('C:\\Users\\nerc-user\\OneDrive - Nexus365\\UTCI_NN\\Main\\data\\UTCI-Test-Data.txt', sep='\t', comment='#')

In [3]:
test['Tr'] = test['Tr-Ta'] + test['Ta']  # convert to absolute MRT

### Run model on the test data to check the output is as expected

In [4]:
test["Offset_NN"] = NN_UTCI(Ta=test['Ta'], Tr=test['Tr'], va=test['va'], rH=test['rH'], oob='nan')

In [5]:
test_analysis = test.copy()
y_true = test_analysis['UTCI']
y_pred = test_analysis['UTCI_polynomial']
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
print(f"RMSE: {rmse:.3f} °C")

RMSE: 2.777 °C


### Test Clamping

In [6]:
NN_UTCI(Ta=70, Tr=28, va=10, rH=100, oob='clamp')

np.float64(80.78778839111328)

In [7]:
NN_UTCI(Ta=70, Tr=28, va=10, rH=110, oob='clamp')

np.float64(80.78778839111328)

### Input shape mismatch - should return an error

In [8]:
NN_UTCI(Ta=np.array([20.0, 25.0]),
        Tr=np.array([25.0]),          # wrong length
        va=np.array([ 1.0,  2.0]),
        rH=np.array([50.0, 60.0]))

ValueError: All inputs must have the same length, got shapes: {1, 2}

### Single input as an array

In [9]:
NN_UTCI(Ta=np.array([20.0]),
            Tr=np.array([25.0]),
            va=np.array([ 1.0]),
            rH=np.array([50.0]))

array([21.4264946])

### One valid row

#### Nan

In [10]:
NN_UTCI(
            Ta=np.array([20.0,  999.0]),
            Tr=np.array([25.0, 1000.0]),
            va=np.array([ 1.0,    1.0]),
            rH=np.array([50.0,   50.0]),
            oob="nan")

array([21.4264946,        nan])

#### clamping

In [11]:
NN_UTCI(
            Ta=np.array([20.0,  999.0]),
            Tr=np.array([25.0, 1000.0]),
            va=np.array([ 1.0,    1.0]),
            rH=np.array([50.0,   50.0]),
            oob="clamp")

array([21.4264946 , 76.80784607])

## Edge cases, One valid, one out of bounds output

In [12]:
NN_UTCI(
            Ta=np.array([-50.0, 50.0]),
            Tr=np.array([0.0, 135.0]),
            va=np.array([  1.0,   35.0]),
            rH=np.array([ 50.0,  50.0]),
            oob="nan",
        )

array([-42.43745804,          nan])